# 🚀 VQA Training — Phase 1 & 2

**Notebook này train mô hình CNN+BiLSTM với:**
- ✅ EfficientNet-B0 pretrained (ImageNet)
- ✅ Image size 224×224
- ✅ AdamW + CosineAnnealingLR
- ✅ Gradient clipping
- ✅ Data augmentation mạnh hơn
- ✅ 20 epoch, early stopping patience=5

## Hướng dẫn:
1. **Runtime → Change runtime type → T4 GPU**
2. Upload file `vqa_project.zip` (xem ô bên dưới)
3. Chạy từng ô theo thứ tự

## Bước 0 — Kiểm tra GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ KHÔNG CÓ GPU — vào Runtime > Change runtime type > T4')
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi

## Bước 1 — Upload và giải nén project

> **Cách tạo file zip:**  
> Trên Windows, mở PowerShell tại thư mục `c:\vqa project` và chạy:
> ```powershell
> Compress-Archive -Path app,src,data,artifacts,requirements.txt,pyproject.toml -DestinationPath vqa_project.zip
> ```
> Sau đó upload file `vqa_project.zip` lên Colab

In [ ]:
from google.colab import files
print('Upload file vqa_project.zip ...')
uploaded = files.upload()

In [ ]:
import zipfile, os

PROJECT_DIR = '/content/vqa_project'
os.makedirs(PROJECT_DIR, exist_ok=True)

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('📁 Project giải nén tại:', PROJECT_DIR)
!ls -la

## Bước 2 — Cài dependencies

In [ ]:
# Colab đã có torch, torchvision — chỉ cần cài thêm timm
!pip install timm==1.0.7 --quiet
print('✅ Dependencies đã cài xong')

## Bước 3 — Kiểm tra data

In [ ]:
import os
from pathlib import Path

data_dir = Path('data')
train_file = data_dir / 'vaq2.0.TrainImages.txt'
dev_file   = data_dir / 'vaq2.0.DevImages.txt'
test_file  = data_dir / 'vaq2.0.TestImages.txt'
img_dir    = data_dir / 'val2014-resised'

print('Train samples :', sum(1 for l in train_file.open() if l.strip()))
print('Dev samples   :', sum(1 for l in dev_file.open()   if l.strip()))
print('Test samples  :', sum(1 for l in test_file.open()  if l.strip()))
print('Images        :', len(list(img_dir.glob('*.jpg'))))

# Kiểm tra class balance
lines = train_file.read_text().splitlines()
yes = sum(1 for l in lines if l.strip().endswith('yes'))
no  = sum(1 for l in lines if l.strip().endswith('no'))
print(f'\nClass balance — YES: {yes} ({yes/len(lines)*100:.1f}%) | NO: {no} ({no/len(lines)*100:.1f}%)')

## Bước 4 — Smoke test nhanh (100 mẫu, 1 epoch)

> Chạy ô này trước để đảm bảo pipeline hoạt động bình thường trước khi train đầy đủ

In [ ]:
import os, tempfile
tmp_dir = tempfile.mkdtemp()
os.environ['VQA_ARTIFACTS_DIR'] = tmp_dir

!python -m src.train_cnn_lstm --pretrained --epochs 1 --limit 100

# Reset về artifacts thật
del os.environ['VQA_ARTIFACTS_DIR']
print('\n✅ Smoke test OK!')

## Bước 5 — TRAIN THẬT SỰ 🚀

⏱️ **Ước tính thời gian:** ~15–25 phút với T4 GPU

In [ ]:
import time
start = time.time()

!python -m src.train_cnn_lstm --pretrained --epochs 20

elapsed = time.time() - start
print(f'\n⏱️ Tổng thời gian train: {elapsed/60:.1f} phút')

## Bước 6 — Xem kết quả & vẽ biểu đồ

In [ ]:
import torch, json
import matplotlib.pyplot as plt

ckpt = torch.load('artifacts/cnn_lstm_model.pt', map_location='cpu', weights_only=True)

print('='*60)
print('📊 KẾT QUẢ TRAINING')
print('='*60)
print(f'Best epoch      : {ckpt["best_epoch"]}')

val_m  = ckpt['metrics']['val']
test_m = ckpt['metrics']['test']
print(f'\n✅ Validation:')
print(f'   Accuracy : {val_m["accuracy"]*100:.2f}%')
print(f'   Macro F1 : {val_m["macro_f1"]*100:.2f}%')
print(f'   Majority baseline: {val_m["majority_baseline_accuracy"]*100:.2f}%')

print(f'\n🧪 Test:')
print(f'   Accuracy : {test_m["accuracy"]*100:.2f}%')
print(f'   Macro F1 : {test_m["macro_f1"]*100:.2f}%')

# Confusion matrix
print(f'\n📉 Confusion Matrix (val):')
cm = val_m['confusion_matrix']
classes = ckpt['classes']
print(f'       {classes}')
for i, row in enumerate(cm):
    print(f'  {classes[i]:3s}  {row}')

# Plot training curves
hist = ckpt['training_history']
epochs = range(1, len(hist['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, hist['train_loss'], 'b-o', label='Train Loss', markersize=4)
axes[0].plot(epochs, hist['val_loss'],   'r-o', label='Val Loss',   markersize=4)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [a*100 for a in hist['val_acc']],      'g-o', label='Val Accuracy', markersize=4)
axes[1].plot(epochs, [a*100 for a in hist['val_macro_f1']], 'm-o', label='Val Macro F1', markersize=4)
axes[1].set_title('Validation Metrics')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('%')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('artifacts/training_curves.png', dpi=150)
plt.show()
print('\n💾 Biểu đồ đã lưu tại artifacts/training_curves.png')

## Bước 7 — Tải model về máy

In [ ]:
import zipfile
from google.colab import files

# Đóng gói artifacts để download
with zipfile.ZipFile('trained_artifacts.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ['artifacts/cnn_lstm_model.pt',
              'artifacts/cnn_lstm_best.pt',
              'artifacts/vocab.json',
              'artifacts/labels.json',
              'artifacts/manifest.json',
              'artifacts/training_curves.png']:
        try:
            z.write(f)
            print(f'✅ Đã thêm: {f}')
        except FileNotFoundError:
            print(f'⚠️  Không tìm thấy: {f}')

print('\n📦 Đang tải về...')
files.download('trained_artifacts.zip')
print('✅ Xong! Giải nén vào thư mục artifacts/ của project.')

## (Tuỳ chọn) — Thử predict ngay trên Colab

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from src.predictor import VQAPredictor
from pathlib import Path

predictor = VQAPredictor(
    checkpoint_path=Path('artifacts/cnn_lstm_model.pt'),
    vocab_path=Path('artifacts/vocab.json'),
    labels_path=Path('artifacts/labels.json'),
)

# Lấy 1 ảnh bất kỳ từ dataset để test
from src.data.loader import load_split
samples = load_split(Path('data/vaq2.0.TestImages.txt'))
sample = samples[0]
img_path = Path('data/val2014-resised') / sample['image_path']

result = predictor.predict(img_path.read_bytes(), sample['question'])
print(f'Question  : {sample["question"]}')
print(f'Answer GT : {sample["answer"]}')
print(f'Predicted : {result["answer"]} (confidence: {result["confidence"]})')
print(f'Probs     : {result["probabilities"]}')